# Counterfactual Shortcut Probe (debugging)

Interactive companion to `model/counterfactual_probe.py` -- same functions, run cell-by-cell so
templates and individual predictions can be inspected before trusting the summary numbers.

Tests the hypothesis in `docs/investigations/2026-07-26-upscale-artifact.md`: keep an image's true
`rgb` content, swap its `fft_mag`/`srm_residual` for another class's averaged template, and check
whether the trained model's prediction follows content or follows the swapped artifact channels.

## Setup

In [ ]:
!pip install -q facenet-pytorch timm kaggle datasets

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
# Mount Drive -- config.py's CHECKPOINT_DIR/DATASET_DIR/EVAL_DIR all resolve under
# DEEPFAKE_DATA_ROOT, so this must be set before any `config`/`model.*` import.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'


In [ ]:
import matplotlib.pyplot as plt

from config import CHECKPOINT_DIR, CLASSES, DEVICE
from model.dataset import ForgeryDataset
from model.eval import load_model
from model.counterfactual_probe import (
    _class_indices,
    extract_templates,
    run_probe,
    summarize,
    print_summary,
    save_probe_results,
    _predict,
)

print("device:", DEVICE)


## Load model + dataset

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "best_model.pt"  # edit if your checkpoint has a different name

model = load_model(str(checkpoint_path))
dataset = ForgeryDataset("val")
indices_by_class = _class_indices(dataset)

{cls: len(idxs) for cls, idxs in indices_by_class.items()}


## Build per-class templates

Small counts here on purpose for a fast debug loop -- bump `TEMPLATE_N` once this looks right.

In [ ]:
TEMPLATE_N = 5  # samples per class averaged into that class's fft/srm template

templates = extract_templates(dataset, indices_by_class, TEMPLATE_N)
{cls: (t["fft_mag"].shape, t["srm_residual"].shape) for cls, t in templates.items()}


## Sanity check: do the templates visually look distinct?

`fft_mag` is single-channel and directly viewable. `srm_residual` is 9-channel (3 SRM kernels x
3 RGB channels) -- shown here is channel 0 only, just to eyeball whether the averaging produced a
sensible (non-degenerate) residual map rather than something that collapsed to near-zero/noise.

In [ ]:
fig, axes = plt.subplots(2, len(CLASSES), figsize=(4 * len(CLASSES), 8))
for col, cls in enumerate(CLASSES):
    axes[0, col].imshow(templates[cls]["fft_mag"][0], cmap="viridis")
    axes[0, col].set_title(f"{cls}: fft_mag template")
    axes[0, col].axis("off")

    axes[1, col].imshow(templates[cls]["srm_residual"][0], cmap="gray")
    axes[1, col].set_title(f"{cls}: srm_residual[0]")
    axes[1, col].axis("off")

fig.tight_layout()


## Run the probe on a few held-out samples

`PROBE_N` samples per class, disjoint from the `TEMPLATE_N` used above (`run_probe` slices
`idxs[TEMPLATE_N : TEMPLATE_N + PROBE_N]` internally) -- baseline (own fft/srm) is printed
alongside every swap so the shift can be read per image.

In [ ]:
PROBE_N = 3

results = run_probe(model, dataset, indices_by_class, templates, PROBE_N, TEMPLATE_N)
len(results)


## Inspect one sample in detail

Re-run with a different index into `results` to look at other probed images.

In [ ]:
r = results[0]
print("path:", r["path"])
print("true_class:", r["true_class"])
print("baseline:", r["baseline"])
print()
for swap_cls, variants in r["swaps"].items():
    tag = "same-class (control)" if swap_cls == r["true_class"] else "cross-class"
    print(f"-- swap -> {swap_cls} [{tag}] --")
    for variant, pred in variants.items():
        print(f"  {variant:>4s}: {pred}")


## Summary

`cross_class - same_class` is the clean signal: `same_class` alone is just the noise floor from
using an averaged template instead of the image's own exact fft/srm values.

In [ ]:
summary = summarize(results)
print_summary(summary)


## Save (optional)

Only run once satisfied with `TEMPLATE_N`/`PROBE_N` -- writes to `EVAL_DIR`.

In [ ]:
save_probe_results(results, summary)
